# NB08 — Preparação de dados para Dashboard Tableau

**Objetivo**: unificar outputs do NB06 (priority queue multimodal) e NB07 (descrições reescritas pelo LLM) com metadados originais do dataset, produzindo dois CSVs prontos para Tableau Desktop.

**Outputs**:
- `results/nb08_tableau/tableau_main.csv` (2997 linhas, test set enriquecido com predições + reescritas)
- `results/nb08_tableau/tableau_full.csv` (14993 linhas, dataset completo para KPIs agregados)

**Audiência do dashboard**: professor avaliador. Duas vistas planeadas:
1. Overview — storytelling científico (comparação de modelos, gradiente de tiers)
2. Priority Queue — demonstração operacional (tabela ordenada por rank)

## Célula 1: Imports e deteção defensiva de PROJECT_ROOT

Aplicamos a lição 1 do NB06: detetar a raiz do projeto procurando marcadores no sistema de ficheiros, em vez de confiar em `Path.cwd()`. Isto evita o problema de o kernel VS Code arrancar na raiz do projeto em vez de `notebooks/`.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root(markers=("data", "notebooks", "results"), max_up=5) -> Path:
    """
    Procura a raiz do projeto subindo a árvore de diretórios a partir do cwd,
    tolerando kernels que arrancam em locais inesperados.
    """
    current = Path.cwd().resolve()
    for _ in range(max_up + 1):
        if all((current / m).exists() for m in markers):
            return current
        if current.parent == current:  # chegou à raiz do filesystem
            break
        current = current.parent
    raise FileNotFoundError(
        f"Não foi possível localizar a raiz do projeto (marcadores: {markers})."
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
OUTPUT_DIR = RESULTS_DIR / "nb08_tableau"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_DIR     : {DATA_DIR}  (exists: {DATA_DIR.exists()})")
print(f"RESULTS_DIR  : {RESULTS_DIR}  (exists: {RESULTS_DIR.exists()})")
print(f"OUTPUT_DIR   : {OUTPUT_DIR}  (exists: {OUTPUT_DIR.exists()})")

PROJECT_ROOT : /Users/rennandamiani/Documents/ISAG/ProjetoFinal_PetFinder
DATA_DIR     : /Users/rennandamiani/Documents/ISAG/ProjetoFinal_PetFinder/data  (exists: True)
RESULTS_DIR  : /Users/rennandamiani/Documents/ISAG/ProjetoFinal_PetFinder/results  (exists: True)
OUTPUT_DIR   : /Users/rennandamiani/Documents/ISAG/ProjetoFinal_PetFinder/results/nb08_tableau  (exists: True)


In [2]:
# --- Metadados originais ---
train_df = pd.read_csv(DATA_DIR / "train" / "train.csv")

# --- Tabelas de lookup ---
breed_labels = pd.read_csv(DATA_DIR / "breed_labels.csv")
color_labels = pd.read_csv(DATA_DIR / "color_labels.csv")
state_labels = pd.read_csv(DATA_DIR / "state_labels.csv")

# --- Outputs dos notebooks anteriores ---
priority_queue = pd.read_csv(
    RESULTS_DIR / "nb06_multimodal" / "priority_queue.csv"
)
rewritten = pd.read_csv(
    RESULTS_DIR / "nb07_llm_rescue" / "rewritten_descriptions.csv"
)

# --- Config dos tiers (para sanity checks mais à frente) ---
with open(RESULTS_DIR / "nb06_multimodal" / "tier_config.json", "r") as f:
    tier_config = json.load(f)

# --- Resumo para confirmação visual ---
print("Shapes das fontes carregadas:")
print(f"  train_df        : {train_df.shape}")
print(f"  breed_labels    : {breed_labels.shape}")
print(f"  color_labels    : {color_labels.shape}")
print(f"  state_labels    : {state_labels.shape}")
print(f"  priority_queue  : {priority_queue.shape}")
print(f"  rewritten       : {rewritten.shape}")
print()
print(f"tier_config keys : {list(tier_config.keys())}")

Shapes das fontes carregadas:
  train_df        : (14993, 24)
  breed_labels    : (307, 3)
  color_labels    : (7, 2)
  state_labels    : (15, 2)
  priority_queue  : (2997, 21)
  rewritten       : (80, 10)

tier_config keys : ['approach', 'rationale', 'quota_high', 'quota_medium', 'cutoff_high', 'cutoff_medium', 'cutoffs_set_on', 'test_metrics']


In [3]:
# Espreitar as colunas de cada fonte para confirmar nomes antes de fazer merges
print("train_df columns:")
print(list(train_df.columns))
print()
print("priority_queue columns:")
print(list(priority_queue.columns))
print()
print("rewritten columns:")
print(list(rewritten.columns))
print()
print("breed_labels head:")
print(breed_labels.head(3))
print()
print("color_labels head:")
print(color_labels.head(3))
print()
print("state_labels head:")
print(state_labels.head(3))

train_df columns:
['Type', 'Name', 'Age', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2', 'Color3', 'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed', 'Sterilized', 'Health', 'Quantity', 'Fee', 'State', 'RescuerID', 'VideoAmt', 'Description', 'PetID', 'PhotoAmt', 'AdoptionSpeed']

priority_queue columns:
['rank', 'PetID', 'tier', 'proba_slow', 'y_proba', 'y_true', 'Name', 'Type', 'Age', 'Breed1', 'Gender', 'Color1', 'MaturitySize', 'FurLength', 'Vaccinated', 'Sterilized', 'Health', 'Quantity', 'Fee', 'PhotoAmt', 'Description']

rewritten columns:
['PetID', 'rank', 'Type_str', 'age_group', 'Age', 'n_words_original', 'n_words_rewritten', 'description_original', 'description_rewritten', 'error']

breed_labels head:
   BreedID  Type         BreedName
0        1     1     Affenpinscher
1        2     1      Afghan Hound
2        3     1  Airedale Terrier

color_labels head:
   ColorID ColorName
0        1     Black
1        2     Brown
2        3    Golden

state_labels head:
   State

In [ ]:
# --- Dicionário de raças ---
# breed_labels tem BreedID, Type, BreedName. Type=1 é Dog, Type=2 é Cat.
breed_map = dict(zip(breed_labels["BreedID"], breed_labels["BreedName"]))
# Adicionar manualmente o código 0 (usado no dataset para "sem segunda raça"/"unknown")
breed_map[0] = "Unknown/None"

# --- Dicionário de cores ---
color_map = dict(zip(color_labels["ColorID"], color_labels["ColorName"]))
color_map[0] = "None"  # código 0 = sem segunda/terceira cor

# --- Dicionário de estados ---
state_map = dict(zip(state_labels["StateID"], state_labels["StateName"]))

# --- Mapeamentos fixos do dataset (não têm CSV) ---
# Ver descrição oficial do dataset PetFinder no Kaggle
type_map = {1: "Dog", 2: "Cat"}
gender_map = {1: "Male", 2: "Female", 3: "Mixed"}  # 3 = grupo misto, usado em ninhadas
maturity_map = {
    0: "Not Specified",
    1: "Small",
    2: "Medium",
    3: "Large",
    4: "Extra Large",
}
fur_length_map = {0: "Not Specified", 1: "Short", 2: "Medium", 3: "Long"}
# Vaccinated/Dewormed/Sterilized usam o mesmo mapa
health_status_map = {1: "Yes", 2: "No", 3: "Not Sure"}
health_condition_map = {
    0: "Not Specified",
    1: "Healthy",
    2: "Minor Injury",
    3: "Serious Injury",
}

# --- Verificar que não vão faltar chaves ---
for col, mapping, name in [
    ("Breed1", breed_map, "breed_map"),
    ("Color1", color_map, "color_map"),
    ("State", state_map, "state_map"),
]:
    missing = set(train_df[col].unique()) - set(mapping.keys())
    if missing:
        print(f"⚠️  {name}: {len(missing)} IDs sem label → {sorted(missing)[:5]}...")
    else:
        print(f"✓ {name}: todos os IDs de train_df[{col}] têm tradução")

print(f"\nTotal de chaves em cada dicionário:")
print(f"  breed_map   : {len(breed_map)}")
print(f"  color_map   : {len(color_map)}")
print(f"  state_map   : {len(state_map)}")
print(f"  type_map    : {len(type_map)}")
print(f"  gender_map  : {len(gender_map)}")

✓ breed_map: todos os IDs de train_df[Breed1] têm tradução
✓ color_map: todos os IDs de train_df[Color1] têm tradução
✓ state_map: todos os IDs de train_df[State] têm tradução

Total de chaves em cada dicionário:
  breed_map   : 308
  color_map   : 8
  state_map   : 15
  type_map    : 2
  gender_map  : 3


In [ ]:
# Começar com cópia do priority_queue para não mutar a fonte original
main = priority_queue.copy()

# --- Verificar se y_proba e proba_slow são redundantes ---
are_equal = (main["y_proba"] == main["proba_slow"]).all()
print(f"y_proba == proba_slow em todas as linhas? {are_equal}")

if are_equal:
    print("→ Descartando y_proba (redundante com proba_slow)")
    main = main.drop(columns=["y_proba"])
else:
    print("⚠️  Não são iguais — manter ambas e investigar depois")
    diff_rows = (main["y_proba"] != main["proba_slow"]).sum()
    print(f"   Linhas diferentes: {diff_rows}")

# --- Sanity checks estruturais ---
assert main["PetID"].is_unique, "PetID não é único em priority_queue!"
assert main["rank"].min() == 1, f"rank não começa em 1 (min={main['rank'].min()})"
assert main["rank"].max() == len(main), (
    f"rank máximo ({main['rank'].max()}) != nº de linhas ({len(main)})"
)
assert set(main["tier"].unique()) == {"High", "Medium", "Low"}, (
    f"Tiers inesperados: {main['tier'].unique()}"
)

print(f"\n✓ main inicial: {main.shape}")
print(f"  Tiers: {main['tier'].value_counts().to_dict()}")


y_proba == proba_slow em todas as linhas? False
⚠️  Não são iguais — manter ambas e investigar depois
   Linhas diferentes: 2997

✓ main inicial: (2997, 21)
  Tiers: {'Low': 1621, 'Medium': 751, 'High': 625}


In [6]:
# Hipótese: proba_slow = 1 - y_proba
sum_probs = main["y_proba"] + main["proba_slow"]

print("Estatísticas de (y_proba + proba_slow):")
print(f"  mean : {sum_probs.mean():.6f}")
print(f"  min  : {sum_probs.min():.6f}")
print(f"  max  : {sum_probs.max():.6f}")
print(f"  std  : {sum_probs.std():.6f}")

# Tolerância para erros de ponto flutuante
are_complementary = np.allclose(sum_probs, 1.0, atol=1e-6)
print(f"\nSão complementares (somam 1)? {are_complementary}")

# Mostrar 5 exemplos para confirmação visual
print("\nExemplos:")
print(main[["PetID", "rank", "tier", "y_proba", "proba_slow"]].head(5))

# Verificar também que o ranking está ordenado por proba_slow (descendente)
is_sorted = main["proba_slow"].is_monotonic_decreasing
print(f"\nRanking ordenado por proba_slow descendente? {is_sorted}")

Estatísticas de (y_proba + proba_slow):
  mean : 1.000000
  min  : 1.000000
  max  : 1.000000
  std  : 0.000000

São complementares (somam 1)? True

Exemplos:
       PetID  rank  tier   y_proba  proba_slow
0  e9eeadf82     1  High  0.417278    0.582722
1  de1f49c9d     2  High  0.418912    0.581088
2  745fd19b8     3  High  0.419002    0.580998
3  7b6629627     4  High  0.422306    0.577694
4  77014fe37     5  High  0.422371    0.577629

Ranking ordenado por proba_slow descendente? True


In [7]:
main = main.drop(columns=["y_proba"])
print(f"✓ main após drop: {main.shape}")
print(f"  Colunas: {list(main.columns)}")

✓ main após drop: (2997, 20)
  Colunas: ['rank', 'PetID', 'tier', 'proba_slow', 'y_true', 'Name', 'Type', 'Age', 'Breed1', 'Gender', 'Color1', 'MaturitySize', 'FurLength', 'Vaccinated', 'Sterilized', 'Health', 'Quantity', 'Fee', 'PhotoAmt', 'Description']


In [ ]:
# --- Colunas categóricas que vão ter versão legível ---

main["Type_str"] = main["Type"].map(type_map)
main["Breed1_str"] = main["Breed1"].map(breed_map)
main["Gender_str"] = main["Gender"].map(gender_map)
main["Color1_str"] = main["Color1"].map(color_map)
main["MaturitySize_str"] = main["MaturitySize"].map(maturity_map)
main["FurLength_str"] = main["FurLength"].map(fur_length_map)
main["Vaccinated_str"] = main["Vaccinated"].map(health_status_map)
main["Sterilized_str"] = main["Sterilized"].map(health_status_map)
main["Health_str"] = main["Health"].map(health_condition_map)

# --- Validar que nenhum map falhou (produziu NaN) ---
str_cols = [
    "Type_str", "Breed1_str", "Gender_str", "Color1_str",
    "MaturitySize_str", "FurLength_str",
    "Vaccinated_str", "Sterilized_str", "Health_str",
]

print("Verificação de NaNs após lookup:")
for col in str_cols:
    n_nan = main[col].isna().sum()
    flag = "✓" if n_nan == 0 else "⚠️"
    print(f"  {flag} {col:20s}: {n_nan} NaN")

# --- Mostrar exemplo de linha para confirmação visual ---
print("\nExemplo (primeira linha):")
sample = main.iloc[0]
for col in ["PetID", "rank", "tier", "Type_str", "Breed1_str",
            "Gender_str", "Color1_str", "Health_str"]:
    print(f"  {col:15s}: {sample[col]}")

Verificação de NaNs após lookup:
  ✓ Type_str            : 0 NaN
  ✓ Breed1_str          : 0 NaN
  ✓ Gender_str          : 0 NaN
  ✓ Color1_str          : 0 NaN
  ✓ MaturitySize_str    : 0 NaN
  ✓ FurLength_str       : 0 NaN
  ✓ Vaccinated_str      : 0 NaN
  ✓ Sterilized_str      : 0 NaN
  ✓ Health_str          : 0 NaN

Exemplo (primeira linha):
  PetID          : e9eeadf82
  rank           : 1
  tier           : High
  Type_str       : Dog
  Breed1_str     : Mixed Breed
  Gender_str     : Female
  Color1_str     : Brown
  Health_str     : Minor Injury


In [9]:
# --- age_group: consistente com NB07 ---
def age_to_group(age_months):
    """Buckets idênticos aos usados no NB07 para sampling estratificado."""
    if age_months <= 6:
        return "Puppy/Kitten (0-6m)"
    elif age_months <= 24:
        return "Young Adult (7-24m)"
    else:
        return "Adult/Senior (25+m)"

main["age_group"] = main["Age"].apply(age_to_group)

# --- Flags binárias (bool) ---
main["has_photo"] = main["PhotoAmt"] > 0
main["is_group"] = main["Quantity"] > 1
main["Fee_is_free"] = main["Fee"] == 0
main["has_name"] = main["Name"].notna() & (main["Name"].str.strip() != "")

# --- tier_order: para ordenação correta no Tableau ---
tier_order_map = {"High": 1, "Medium": 2, "Low": 3}
main["tier_order"] = main["tier"].map(tier_order_map)

# --- Predicted class: derivada do tier (útil para KPIs) ---
# High = modelo considera "alto risco de adoção lenta"
main["predicted_slow"] = main["tier"] == "High"

# --- y_true_str: para legibilidade no Tableau ---
main["y_true_str"] = main["y_true"].map({0: "Slow", 1: "Fast"})

# --- Validação ---
print("Distribuições das features derivadas:")
print(f"\nage_group:")
print(main["age_group"].value_counts().to_dict())
print(f"\nhas_photo: {main['has_photo'].sum()} True ({main['has_photo'].mean()*100:.1f}%)")
print(f"is_group: {main['is_group'].sum()} True ({main['is_group'].mean()*100:.1f}%)")
print(f"Fee_is_free: {main['Fee_is_free'].sum()} True ({main['Fee_is_free'].mean()*100:.1f}%)")
print(f"has_name: {main['has_name'].sum()} True ({main['has_name'].mean()*100:.1f}%)")

print(f"\ntier_order alinhado com tier?")
print(main.groupby("tier")["tier_order"].first().to_dict())

print(f"\ny_true_str:")
print(main["y_true_str"].value_counts().to_dict())

print(f"\n✓ main após derived fields: {main.shape}")

Distribuições das features derivadas:

age_group:
{'Puppy/Kitten (0-6m)': 2093, 'Young Adult (7-24m)': 635, 'Adult/Senior (25+m)': 269}

has_photo: 2933 True (97.9%)
is_group: 628 True (21.0%)
Fee_is_free: 2574 True (85.9%)
has_name: 2779 True (92.7%)

tier_order alinhado com tier?
{'High': 1, 'Low': 3, 'Medium': 2}

y_true_str:
{'Fast': 1506, 'Slow': 1491}

✓ main após derived fields: (2997, 37)


In [ ]:
# --- Preparar rewritten para o merge ---
rewritten_for_merge = rewritten[[
    "PetID",
    "n_words_original",
    "n_words_rewritten",
    "description_original",
    "description_rewritten",
]].copy()

# Sanity: confirmar que PetIDs do rewritten existem todos em main
rewritten_petids = set(rewritten_for_merge["PetID"])
main_petids = set(main["PetID"])
missing_in_main = rewritten_petids - main_petids
print(f"PetIDs de rewritten ausentes em main: {len(missing_in_main)}")
assert len(missing_in_main) == 0, "Reescritas referem PetIDs fora do test set!"

# --- Merge: left join (preserva as 2997 linhas do main) ---
main = main.merge(rewritten_for_merge, on="PetID", how="left")

# --- Flag: tem descrição reescrita? ---
main["has_rewritten_description"] = main["description_rewritten"].notna()

# --- Campo derivado: delta de palavras (só faz sentido quando há reescrita) ---
main["length_delta_words"] = (
    main["n_words_rewritten"] - main["n_words_original"]
)

# --- Validação ---
n_rewritten = main["has_rewritten_description"].sum()
print(f"\n✓ Animais com reescrita: {n_rewritten} (esperado: 80)")
assert n_rewritten == 80, f"Esperavam-se 80 reescritas, encontradas {n_rewritten}"

# Distribuição dos reescritos por tier (confirma estratificação do NB07)
print(f"\nReescritas por tier:")
print(
    main[main["has_rewritten_description"]]
    .groupby("tier")
    .size()
    .to_dict()
)

# Distribuição por Type
print(f"\nReescritas por Type:")
print(
    main[main["has_rewritten_description"]]
    .groupby("Type_str")
    .size()
    .to_dict()
)

# Estatísticas do length_delta
delta = main.loc[main["has_rewritten_description"], "length_delta_words"]
print(f"\nlength_delta_words (só reescritos):")
print(f"  mean : {delta.mean():+.1f} palavras")
print(f"  min  : {delta.min():+.0f}")
print(f"  max  : {delta.max():+.0f}")
print(f"  positivos (expansão): {(delta > 0).sum()}")
print(f"  negativos (compressão): {(delta < 0).sum()}")

print(f"\n✓ main após merge com rewritten: {main.shape}")


PetIDs de rewritten ausentes em main: 0

✓ Animais com reescrita: 80 (esperado: 80)

Reescritas por tier:
{'High': 80}

Reescritas por Type:
{'Cat': 40, 'Dog': 40}

length_delta_words (só reescritos):
  mean : -12.4 palavras
  min  : -284
  max  : +55
  positivos (expansão): 42
  negativos (compressão): 38

✓ main após merge com rewritten: (2997, 43)


In [11]:
# --- Base ---
full = train_df.copy()

# --- Target binário (consistente com todos os notebooks) ---
full["AdoptionFast"] = (full["AdoptionSpeed"] <= 2).astype(int)
full["AdoptionFast_str"] = full["AdoptionFast"].map({0: "Slow", 1: "Fast"})

# --- Lookups (idênticos aos do main) ---
full["Type_str"] = full["Type"].map(type_map)
full["Breed1_str"] = full["Breed1"].map(breed_map)
full["Gender_str"] = full["Gender"].map(gender_map)
full["Color1_str"] = full["Color1"].map(color_map)
full["State_str"] = full["State"].map(state_map)
full["MaturitySize_str"] = full["MaturitySize"].map(maturity_map)
full["FurLength_str"] = full["FurLength"].map(fur_length_map)
full["Vaccinated_str"] = full["Vaccinated"].map(health_status_map)
full["Dewormed_str"] = full["Dewormed"].map(health_status_map)
full["Sterilized_str"] = full["Sterilized"].map(health_status_map)
full["Health_str"] = full["Health"].map(health_condition_map)

# --- Campos derivados (idênticos aos do main para consistência) ---
full["age_group"] = full["Age"].apply(age_to_group)
full["has_photo"] = full["PhotoAmt"] > 0
full["is_group"] = full["Quantity"] > 1
full["Fee_is_free"] = full["Fee"] == 0
full["has_name"] = full["Name"].notna() & (full["Name"].str.strip() != "")

# --- Validação ---
# Verificar se todos os NaN mappings falharam
str_cols_full = [
    "Type_str", "Breed1_str", "Gender_str", "Color1_str", "State_str",
    "MaturitySize_str", "FurLength_str",
    "Vaccinated_str", "Dewormed_str", "Sterilized_str", "Health_str",
]
print("NaNs em colunas _str:")
for col in str_cols_full:
    n_nan = full[col].isna().sum()
    flag = "✓" if n_nan == 0 else "⚠️"
    print(f"  {flag} {col:20s}: {n_nan}")

# Confirmar binarização do target
print(f"\nAdoptionFast (target binário):")
print(full["AdoptionFast"].value_counts().to_dict())
print(f"  Taxa de adoção rápida: {full['AdoptionFast'].mean()*100:.2f}%")

# Confirmar distribuições
print(f"\nDistribuição global por Type:")
print(full["Type_str"].value_counts().to_dict())

print(f"\nDistribuição por age_group:")
print(full["age_group"].value_counts().to_dict())

print(f"\nhas_photo: {full['has_photo'].sum()} ({full['has_photo'].mean()*100:.1f}%)")
print(f"is_group: {full['is_group'].sum()} ({full['is_group'].mean()*100:.1f}%)")

print(f"\n✓ full: {full.shape}")

NaNs em colunas _str:
  ✓ Type_str            : 0
  ✓ Breed1_str          : 0
  ✓ Gender_str          : 0
  ✓ Color1_str          : 0
  ✓ State_str           : 0
  ✓ MaturitySize_str    : 0
  ✓ FurLength_str       : 0
  ✓ Vaccinated_str      : 0
  ✓ Dewormed_str        : 0
  ✓ Sterilized_str      : 0
  ✓ Health_str          : 0

AdoptionFast (target binário):
{1: 7537, 0: 7456}
  Taxa de adoção rápida: 50.27%

Distribuição global por Type:
{'Dog': 8132, 'Cat': 6861}

Distribuição por age_group:
{'Puppy/Kitten (0-6m)': 10214, 'Young Adult (7-24m)': 3238, 'Adult/Senior (25+m)': 1541}

has_photo: 14652 (97.7%)
is_group: 3428 (22.9%)

✓ full: (14993, 42)


In [ ]:
# --- Check 1: test set em main tem os mesmos PetIDs que em full ---
main_petids = set(main["PetID"])
full_petids = set(full["PetID"])

overlap = main_petids & full_petids
main_not_in_full = main_petids - full_petids

print(f"Check 1 — PetIDs de main presentes em full:")
print(f"  Overlap: {len(overlap)}/{len(main_petids)}")
print(f"  main PetIDs ausentes em full: {len(main_not_in_full)}")
assert len(main_not_in_full) == 0, "Há PetIDs em main que não existem em full!"

# --- Check 2: tiers em main usam cutoffs consistentes com o NB06 ---
cutoff_high = tier_config["cutoff_high"]
cutoff_medium = tier_config["cutoff_medium"]
print(f"\nCheck 2 — Cutoffs dos tiers (do tier_config.json):")
print(f"  cutoff_high   : {cutoff_high:.6f}")
print(f"  cutoff_medium : {cutoff_medium:.6f}")

# Verificar que proba_slow nos animais High >= cutoff_high
high_mask = main["tier"] == "High"
min_proba_high = main.loc[high_mask, "proba_slow"].min()
print(f"  min proba_slow em tier High: {min_proba_high:.6f}")
assert min_proba_high >= cutoff_high - 1e-6, (
    f"Tier High tem proba < cutoff_high ({min_proba_high} < {cutoff_high})"
)

# --- Check 3: age_group consistente em main vs full para o mesmo PetID ---
check3 = main[["PetID", "age_group"]].merge(
    full[["PetID", "age_group"]].rename(columns={"age_group": "age_group_full"}),
    on="PetID",
    how="left",
)
inconsistent = (check3["age_group"] != check3["age_group_full"]).sum()
print(f"\nCheck 3 — age_group consistente entre main e full:")
print(f"  Linhas com inconsistência: {inconsistent}")
assert inconsistent == 0, "age_group diverge entre main e full!"

# --- Check 4: sem duplicatas de PetID em nenhum dos dois ---
print(f"\nCheck 4 — Duplicatas de PetID:")
print(f"  main : {main['PetID'].duplicated().sum()}")
print(f"  full : {full['PetID'].duplicated().sum()}")
assert main["PetID"].duplicated().sum() == 0
assert full["PetID"].duplicated().sum() == 0

# --- Check 5: encoding das descrições é UTF-8 safe ---
sample_descs = main["Description"].dropna().sample(min(50, len(main)), random_state=42)
encoding_ok = True
for desc in sample_descs:
    try:
        desc.encode("utf-8").decode("utf-8")
    except (UnicodeDecodeError, UnicodeEncodeError):
        encoding_ok = False
        break

print(f"\nCheck 5 — Encoding UTF-8 nas descrições: {'✓ OK' if encoding_ok else '⚠️ Problema'}")

# --- Check 6: tamanho total das descrições (só para contexto) ---
rewritten_with_desc = main[main["has_rewritten_description"]]
mean_orig = rewritten_with_desc["n_words_original"].mean()
mean_rew = rewritten_with_desc["n_words_rewritten"].mean()
print(f"\nCheck 6 — Reescritas (contexto para relatório):")
print(f"  Média original   : {mean_orig:.1f} palavras")
print(f"  Média reescrita  : {mean_rew:.1f} palavras")
print(f"  Diferença média  : {mean_rew - mean_orig:+.1f}")

print(f"\n✓ Todos os sanity checks passaram")

Check 1 — PetIDs de main presentes em full:
  Overlap: 2997/2997
  main PetIDs ausentes em full: 0

Check 2 — Cutoffs dos tiers (do tier_config.json):
  cutoff_high   : 0.519788
  cutoff_medium : 0.477845
  min proba_slow em tier High: 0.519835

Check 3 — age_group consistente entre main e full:
  Linhas com inconsistência: 0

Check 4 — Duplicatas de PetID:
  main : 0
  full : 0

Check 5 — Encoding UTF-8 nas descrições: ✓ OK

Check 6 — Reescritas (contexto para relatório):
  Média original   : 72.9 palavras
  Média reescrita  : 60.4 palavras
  Diferença média  : -12.5

✓ Todos os sanity checks passaram


In [ ]:
# --- Paths de output ---
main_path = OUTPUT_DIR / "tableau_main.csv"
full_path = OUTPUT_DIR / "tableau_full.csv"

# --- Export ---
main.to_csv(main_path, index=False, encoding="utf-8-sig")
full.to_csv(full_path, index=False, encoding="utf-8-sig")

# --- Verificação ---
main_size_kb = main_path.stat().st_size / 1024
full_size_kb = full_path.stat().st_size / 1024

print(f"✓ tableau_main.csv guardado")
print(f"  Path   : {main_path}")
print(f"  Size   : {main_size_kb:.1f} KB")
print(f"  Shape  : {main.shape}")

print(f"\n✓ tableau_full.csv guardado")
print(f"  Path   : {full_path}")
print(f"  Size   : {full_size_kb:.1f} KB")
print(f"  Shape  : {full.shape}")

# --- Round-trip check: ler de volta e confirmar integridade ---
main_reloaded = pd.read_csv(main_path)
full_reloaded = pd.read_csv(full_path)

assert main_reloaded.shape == main.shape, "main: shape mudou no round-trip!"
assert full_reloaded.shape == full.shape, "full: shape mudou no round-trip!"
assert list(main_reloaded.columns) == list(main.columns), "main: colunas mudaram!"
assert list(full_reloaded.columns) == list(full.columns), "full: colunas mudaram!"

print(f"\n✓ Round-trip check passou (leitura do CSV = DataFrame original)")

✓ tableau_main.csv guardado
  Path   : /Users/rennandamiani/Documents/ISAG/ProjetoFinal_PetFinder/results/nb08_tableau/tableau_main.csv
  Size   : 1741.0 KB
  Shape  : (2997, 43)

✓ tableau_full.csv guardado
  Path   : /Users/rennandamiani/Documents/ISAG/ProjetoFinal_PetFinder/results/nb08_tableau/tableau_full.csv
  Size   : 8328.3 KB
  Shape  : (14993, 42)

✓ Round-trip check passou (leitura do CSV = DataFrame original)


In [14]:
# --- Preview do tableau_main ---
print("=" * 60)
print("TABLEAU_MAIN — primeiras 3 linhas, colunas-chave")
print("=" * 60)

preview_cols_main = [
    "rank", "PetID", "tier", "proba_slow", "y_true_str",
    "Type_str", "Breed1_str", "age_group",
    "has_photo", "has_rewritten_description",
]
print(main[preview_cols_main].head(3).to_string(index=False))

print("\nColunas completas de tableau_main:")
for i, col in enumerate(main.columns, 1):
    print(f"  {i:2d}. {col}")

# --- Preview do tableau_full ---
print("\n" + "=" * 60)
print("TABLEAU_FULL — primeiras 3 linhas, colunas-chave")
print("=" * 60)

preview_cols_full = [
    "PetID", "Type_str", "Breed1_str", "age_group",
    "AdoptionSpeed", "AdoptionFast_str", "has_photo", "State_str",
]
print(full[preview_cols_full].head(3).to_string(index=False))

print("\nColunas completas de tableau_full:")
for i, col in enumerate(full.columns, 1):
    print(f"  {i:2d}. {col}")

# --- Exemplo de reescrita para referência rápida ---
print("\n" + "=" * 60)
print("EXEMPLO DE REESCRITA (para sanity visual)")
print("=" * 60)
example = main[main["has_rewritten_description"]].iloc[0]
print(f"PetID: {example['PetID']} | Rank: {example['rank']} | Tier: {example['tier']}")
print(f"Type: {example['Type_str']} | Age: {example['Age']}m | Breed: {example['Breed1_str']}")
print(f"\n[Original — {example['n_words_original']} palavras]:")
print(example["description_original"][:300] + ("..." if len(str(example["description_original"])) > 300 else ""))
print(f"\n[Reescrita — {example['n_words_rewritten']} palavras]:")
print(example["description_rewritten"][:300] + ("..." if len(str(example["description_rewritten"])) > 300 else ""))

TABLEAU_MAIN — primeiras 3 linhas, colunas-chave
 rank     PetID tier  proba_slow y_true_str Type_str  Breed1_str           age_group  has_photo  has_rewritten_description
    1 e9eeadf82 High    0.582722       Slow      Dog Mixed Breed Young Adult (7-24m)       True                       True
    2 de1f49c9d High    0.581088       Slow      Dog Mixed Breed Puppy/Kitten (0-6m)       True                       True
    3 745fd19b8 High    0.580998       Slow      Dog Mixed Breed Young Adult (7-24m)       True                       True

Colunas completas de tableau_main:
   1. rank
   2. PetID
   3. tier
   4. proba_slow
   5. y_true
   6. Name
   7. Type
   8. Age
   9. Breed1
  10. Gender
  11. Color1
  12. MaturitySize
  13. FurLength
  14. Vaccinated
  15. Sterilized
  16. Health
  17. Quantity
  18. Fee
  19. PhotoAmt
  20. Description
  21. Type_str
  22. Breed1_str
  23. Gender_str
  24. Color1_str
  25. MaturitySize_str
  26. FurLength_str
  27. Vaccinated_str
  28. Sterilized_s